In [ ]:
import importlib

import browser_manager
import helpers
import entities
import models

importlib.reload(browser_manager)
importlib.reload(helpers)
importlib.reload(entities)
# importlib.reload(models)

from browser_manager import BrowserManager
from helpers import Helpers
from entities import Element
from models import model, strong_model

from playwright.async_api import async_playwright
from langchain_core.tools import tool
import json
from dataclasses import dataclass
from typing import Any

from deepagents import create_deep_agent, DeepAgentState
from pywin.framework.toolmenu import tools
from sqlalchemy.sql.base import elements

SYSTEM_PROMPT = """
You are a web browser agent.
Use `observe_page` once to inspect the current page and available interactions before deciding what to do next.

Given the user's task and the observed page state, determine the next interaction or sequence of interactions that makes useful progress toward the task.
Guidelines:
- Base all interactions only on the observed page state and provided context.
- Reference only elements and information present in the observation.
- Use element indices exactly as provided.
- Do not invent page content, elements, or available actions.
- Prefer the smallest useful sequence of interactions.
- You may return multiple interactions when they can all be determined from the current observed state.
- Do not plan interactions that depend on the result of an earlier interaction unless that result is already known.

After observing the page, use the `task` tool to delegate browser actions. Set `subagent_type` to `steps-executor`. Set the task tool's `description` argument to a JSON-serialized string containing exactly this structure:
{
  "steps": [
    {"action": "click", "element_index": 4},
    {"action": "fill", "element_index": 1, "value": "text to enter"}
  ]
}
The `description` argument must contain only that JSON object. Do not summarize the steps in prose. Do not return the execution plan directly to the user instead of calling the `task` tool.
Use only the actions `click`, `fill`, `select`, or `press`. The `value` field is required for every action except `click`. Use exact indices from the latest observation, and make any page-changing action the final step. Do not include explanations, Markdown, an expected URL, or an observation ID in the JSON plan.

Never assume an interaction succeeded or that the page changed unless explicitly indicated by a subsequent observation or provided context.
Never call `observe_page` twice consecutively for the same URL or unchanged page. Only observe again after another browser action explicitly reports that the page may have changed.
If the observation does not provide enough information, or no available tool can perform the next interaction, stop and clearly explain what is blocking progress instead of observing again or guessing.
"""


@tool
async def observe_page(url: str) -> str:
    """
    Inspect the current page and return a compact snapshot for reasoning.

    The snapshot contains:
    - page URL and title
    - indexed interactive elements
    - visible page text
    - an estimate of how much content remains below the viewport

    Interactive elements are referenced by their [index] in subsequent
    browser action tools.

    Use this tool to understand the page and decide the next browser action.
    """
    browser_manager = BrowserManager()
    return await browser_manager.observe_page(url)


In [ ]:
from langchain.tools import tool


@tool
async def click(element_index: int) -> str:
    """Click an element in the browser by its element index."""
    browser_manager = BrowserManager()
    await browser_manager.click(element_index)
    return f"Clicked element [{element_index}]"


@tool
async def fill(element_index: int, value: str) -> str:
    """Fill an input element in the browser with a value."""
    browser_manager = BrowserManager()
    await browser_manager.fill(element_index, value)
    return f"Filled element [{element_index}]"


@tool
async def select(element_index: int, value: str) -> str:
    """Select an option from a browser select element."""
    browser_manager = BrowserManager()
    await browser_manager.select(element_index, value)
    return f"Selected an option in element [{element_index}]"


@tool
async def press(element_index: int, value: str) -> str:
    """Press a keyboard key or key combination on a browser element."""
    browser_manager = BrowserManager()
    await browser_manager.press(element_index, value)
    return f"Pressed {value} on element [{element_index}]"

In [ ]:
from langchain_quickjs import CodeInterpreterMiddleware

STEPS_EXECUTOR_SUBAGENT_SYSTEM_PROMPT = """
You are a web browser agent that executes the steps provided to you.
You have access to the following browser tools: click, fill, select, and press.
The task will contain a list of steps. Execute each step in order using the
corresponding tool and the provided arguments.
The task will look like this:
{
  "steps": [
    {"action": "click", "element_index": 4},
    {"action": "fill", "element_index": 1, "value": "text to enter"},
    {"action": "select", "element_index": 2, "value": "option_value"},
    {"action": "press", "element_index": 3, "value": "Enter"}
  ]
}

The eval tool supports Programmatic Tool Calling (PTC): JavaScript running
inside eval() can call the browser tools through tools.click(),
tools.fill(), tools.select(), and tools.press().
Prefer a single eval() call that executes all provided steps sequentially in
JavaScript. Keep intermediate tool results inside JavaScript variables rather
than returning them to the model between steps.
For each step:
- Call the tool whose name matches the "action".
- Pass "element_index" to the tool.
- If "value" is present, pass it as the value argument.
- Await each tool call before executing the next step.
- Execute the steps in the exact order provided.

Do not skip, reorder, modify, or invent steps.
After all steps are completed, return a short confirmation that the steps
were executed successfully.
"""

# steps_executor_subagent = create_deep_agent(
#     model=model,
#     system_prompt=STEPS_EXECUTOR_SUBAGENT_SYSTEM_PROMPT,
#     middleware=[CodeInterpreterMiddleware(ptc=["click", "fill", "select", "press"])],
#     tools=[click, fill, select, press]
# )

steps_executor = {
    "name": "steps-executor",
    "description": (
        "Execute browser interaction steps in order. The task description must be "
        "a JSON object containing a steps array; never accept or infer steps from prose."
    ),
    "system_prompt": STEPS_EXECUTOR_SUBAGENT_SYSTEM_PROMPT,
    "tools": [click, fill, select, press],
    "model": model,
    "middleware": [CodeInterpreterMiddleware(ptc=["click", "fill", "select", "press"])],
}


In [13]:
agent = create_deep_agent(model=model, system_prompt=SYSTEM_PROMPT, tools=[observe_page],
                          subagents=[steps_executor])
result = await agent.ainvoke({"messages": [{"role": "user", "content": "Perform user login in https://backlogr.dev with email test@backlogr.com and password test12345"}]})
print(result["messages"][-1].content)


waiting for domcontentloaded
waiting for load
waiting for domcontentloaded
waiting for load
Login attempt completed. The current page shows no interactive elements or visible content, which likely means the page loaded a new view (perhaps a dashboard) or an error/redirect occurred that didn’t render interactive controls in this view.

What would you like to do next?
- I can try reloading the page or navigate to the dashboard to confirm login success.
- I can attempt the login again or check for a different URL after login.
- If you have a specific post-login URL (e.g., /dashboard), I can try opening that directly.


In [ ]:
browser_manager = BrowserManager()
await browser_manager.close()